In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip uninstall -y ragas langchain langchain_community langchain_openai

!pip install "ragas==0.4.3" "langchain==0.3.30" "langchain_community==0.3.31" "langchain_openai==0.3.35" --no-cache-dir

In [ ]:
!pip install "langchain_ollama<0.3" --no-cache-dir

In [ ]:
!pip install rapidfuzz

In [ ]:
!cp "/kaggle/input/datasets/""/contexual-emb/kaggle_contexual_embedding.py" .
!cp "/kaggle/input/datasets/""/contexual-emb/all_chunks_contextual_checkpoint.pkl" .

In [ ]:
# Güncel ve açıklamalı .env dosyasını Colab'de oluşturur:
env_content = """# ==============================================================================
# AZURE OPENAI AYARLARI (Ragas Soru-Cevap & Değerlendirme LLM Bağlantısı)
# ==============================================================================
AZURE_OPENAI_API_KEY = ""
AZURE_OPENAI_ENDPOINT=""
AZURE_OPENAI_API_VERSION=2024-02-15-preview
AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4.1-mini
CHECKPOINT_PATH=/kaggle/working/all_chunks_contextual_checkpoint.pkl
INPUT_RESULTS_PATH=rag_results_basic_qwen.csv

# Puanlanmış sonuçların kaydedileceği hedef dosya:
OUTPUT_EVAL_PATH=outputs/rag_results_basic_qwen_evaluated.csv

"""

with open(".env", "w", encoding="utf-8") as f:
    f.write(env_content)

print("[✓] .env dosyası yeni RAG parametreleriyle başarıyla güncellendi!")

In [ ]:
!pip install chromadb

In [ ]:
!python /kaggle/working/kaggle_contexual_embedding.py

In [ ]:
# whats the role of americn academy of pediatrics in diabetes care?

In [ ]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# .env dosyasındaki değişkenleri yükle
load_dotenv()

# ==============================================================================
# 1. ADIM: Azure OpenAI ve Embedding Modelini Tanımla
# ==============================================================================
azure_client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)
DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

print("[*] Nomic Embed modeli yükleniyor...")
embeddings = HuggingFaceEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1.5",
    model_kwargs={
        "trust_remote_code": True, 
        "device": "cuda" if os.system("nvidia-smi > /dev/null 2>&1") == 0 else "cpu"
    }
)

# ==============================================================================
# 2. ADIM: Kaydedilmiş Chroma DB'yi Yükle ve Arama Yap (Retrieval)
# ==============================================================================
db_dir = "chroma_db_contextual"
print(f"[*] '{db_dir}' dizinindeki veritabanı yükleniyor...")
db = Chroma(persist_directory=db_dir, embedding_function=embeddings)

# Test edilecek klinik soru
#query = "whats the role of americn academy of pediatrics in diabetes care?"
query = "What are the recommended management strategies for children and adolescents with diabetes to address hypertension and nephropathy?"
nomic_query = f"search_query: {query}"

print(f"\n🔎 RAG SORGUSU: '{query}'")
retrieved_docs = db.similarity_search(nomic_query, k=5)

# Çekilen dokümanları bağlam olarak hazırla ve kaynak listesini kaydet
context_parts = []
source_metadata_list = []

for idx, doc in enumerate(retrieved_docs, 1):
    clean_content = doc.page_content.replace("search_document: ", "", 1)
    source = doc.metadata.get('source')
    h1 = doc.metadata.get('header_1')
    h2 = doc.metadata.get('header_2', '')
    
    context_parts.append(f"--- Kaynak [{idx}]: Dosya: {source} | Başlık: {h1} > {h2} ---\n{clean_content}")
    
    source_metadata_list.append({
        "id": idx,
        "source": source,
        "header_1": h1,
        "header_2": h2,
        "snippet": clean_content[:]
    })

combined_context = "\n\n".join(context_parts)

# ==============================================================================
# 3. ADIM: Güvenli ve Kaynak Odaklı Prompt Tasarımı
# ==============================================================================
prompt = f"""
You are a rigorous clinical decision support assistant.

Your task is to answer the user's question using ONLY the information
contained in the provided official medical guideline contexts.

CRITICAL RULES:

1. Do NOT use your pre-trained knowledge or external information.
   Your answer must be supported by the provided contexts.

2. Carefully inspect ALL provided sources before deciding that the
   answer is not available.

3. If the question is directly answered by one or more sources,
   provide that answer clearly and concisely.

4. If the answer is partially supported by the contexts, provide only
   the information that is supported by the contexts. Do not fill the
   missing information using your own knowledge.

5. If the provided contexts genuinely do not contain enough information
   to answer the question, say:
   "The provided guidelines do not contain enough information to answer
   this question."

6. Every factual statement in your answer MUST be followed by the
   source number that supports it, using this format:
   [Source 1], [Source 2], etc.

7. Do NOT cite a source unless the information in that source actually
   supports the statement.

8. Prefer the most direct evidence available in the retrieved context.
   Do not replace a directly stated fact with a general explanation.

[PROVIDED OFFICIAL GUIDELINE CONTEXTS]

{combined_context}

[USER QUESTION]

{query}

[CLINICAL ANSWER]
"""

print("[*] Azure OpenAI ile klinik yanıt üretiliyor...")
response = azure_client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.0,  # Halüsinasyonu minimuma indirmek için sıfıra yakın tutuyoruz
    max_tokens=500
)

answer = response.choices[0].message.content.strip()

# ==============================================================================
# 4. ADIM: Sonucu ve Kanıt Olarak Kullanılan Kaynakları Ekrana Bas
# ==============================================================================
print("\n" + "="*70)
print("               AZURE OPENAI + RAG KONTROLLÜ YANIT")
print("="*70)
print(answer)
print("="*70)

print("\n🔍 MODELİN BİLGİYİ ÇEKTİĞİ GERÇEK VERİTABANI KAYNAKLARI:")
print("-" * 70)
for meta in source_metadata_list:
    print(f"[Kaynak {meta['id']}]")
    print(f"  - Dosya Adı     : {meta['source']}")
    print(f"  - Bölüm (H1)    : {meta['header_1']}")
    print(f"  - Alt Bölüm (H2): {meta['header_2']}")
    print(f"  - Metin Parçası : {meta['snippet']}...")
    print()
print("="*70)

## reranker

In [ ]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from sentence_transformers import CrossEncoder

# .env dosyasındaki değişkenleri yükle
load_dotenv()

# ==============================================================================
# 1. ADIM: Azure OpenAI, Embedding ve Reranker (Cross-Encoder) Modelini Yükle
# ==============================================================================
azure_client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)
DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

print("[*] Nomic Embed modeli yükleniyor...")
embeddings = HuggingFaceEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1.5",
    model_kwargs={
        "trust_remote_code": True, 
        "device": "cuda" if os.system("nvidia-smi > /dev/null 2>&1") == 0 else "cpu"
    }
)

print("[*] Reranker modeli (ms-marco-MiniLM-L-6-v2) yükleniyor...")
# Cross-Encoder, soru ve doküman ikilisini birlikte değerlendirerek en hassas puanlamayı yapar
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# ==============================================================================
# 2. ADIM: Chroma DB'den Geniş Aday Kümesi Çek (Retrieval - Top 10)
# ==============================================================================
db_dir = "chroma_db_contextual"
print(f"[*] '{db_dir}' dizinindeki veritabanı yükleniyor...")
db = Chroma(persist_directory=db_dir, embedding_function=embeddings)

#query = "whats the role of americn academy of pediatrics in diabetes care?"
query = "What are the recommended management strategies for children and adolescents with diabetes to address hypertension and nephropathy?"

nomic_query = f"search_query: {query}"

print(f"\n🔎 RAG SORGUSU: '{query}'")
# İlk aşamada geniş bir havuz çekiyoruz (örn: k=10)
initial_docs = db.similarity_search(nomic_query, k=20)

# ==============================================================================
# 3. ADIM: RERANKER — Adayları Yeniden Puanla ve Sırala
# ==============================================================================
print("[*] Çekilen dokümanlar Reranker ile yeniden sıralanıyor...")

# Cross-Encoder için (soru, doküman_metni) çiftleri oluştur
pairs = []
for doc in initial_docs:
    clean_text = doc.page_content.replace("search_document: ", "", 1)
    pairs.append([query, clean_text])

# Skorları hesapla
scores = reranker_model.predict(pairs)

# Dokümanları skorlarıyla birlikte eşleştir ve büyükten küçüğe sırala
scored_docs = list(zip(initial_docs, scores))
scored_docs.sort(key=lambda x: x[1], reverse=True)

# En iyi Rerank edilmiş üst dokümanları seç (örn: Top 3)
top_k_reranked = scored_docs[:5]

# ==============================================================================
# 4. ADIM: Rerank Edilmiş Bağlamı Hazırla
# ==============================================================================
context_parts = []
source_metadata_list = []

for idx, (doc, score) in enumerate(top_k_reranked, 1):
    clean_content = doc.page_content.replace("search_document: ", "", 1)
    source = doc.metadata.get('source')
    h1 = doc.metadata.get('header_1')
    h2 = doc.metadata.get('header_2', '')
    
    context_parts.append(f"--- Kaynak [{idx}] (Rerank Skor: {score:.4f}): Dosya: {source} | Başlık: {h1} > {h2} ---\n{clean_content}")
    
    source_metadata_list.append({
        "id": idx,
        "score": float(score),
        "source": source,
        "header_1": h1,
        "header_2": h2,
        "snippet": clean_content
    })

combined_context = "\n\n".join(context_parts)

# ==============================================================================
# 5. ADIM: Azure OpenAI ile Yanıt Üretimi
# ==============================================================================
prompt = f"""
You are a rigorous clinical decision support assistant.

Your task is to answer the user's question using ONLY the information
contained in the provided official medical guideline contexts.

CRITICAL RULES:

1. Do NOT use your pre-trained knowledge or external information.
   Your answer must be supported by the provided contexts.

2. Carefully inspect ALL provided sources before deciding that the
   answer is not available.

3. If the question is directly answered by one or more sources,
   provide that answer clearly and concisely.

4. If the answer is partially supported by the contexts, provide only
   the information that is supported by the contexts. Do not fill the
   missing information using your own knowledge.

5. If the provided contexts genuinely do not contain enough information
   to answer the question, say:
   "The provided guidelines do not contain enough information to answer
   this question."

6. Every factual statement in your answer MUST be followed by the
   source number that supports it, using this format:
   [Source 1], [Source 2], etc.

7. Do NOT cite a source unless the information in that source actually
   supports the statement.

8. Prefer the most direct evidence available in the retrieved context.
   Do not replace a directly stated fact with a general explanation.

[PROVIDED OFFICIAL GUIDELINE CONTEXTS]


{combined_context}



[USER QUESTION]



{query}



[CLINICAL ANSWER]

"""

print("[*] Azure OpenAI ile final klinik yanıt üretiliyor...")
response = azure_client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.0,
    max_tokens=500
)

answer = response.choices[0].message.content.strip()

# ==============================================================================
# 6. ADIM: Sonucu ve Rerank Edilmiş Kaynakları Ekrana Bas
# ==============================================================================
print("\n" + "="*70)
print("       AZURE OPENAI + CONTEXTUAL RAG + RERANKER KONTROLLÜ YANIT")
print("="*70)
print(answer)
print("="*70)

print("\n🔍 RERANKER TARAFINDAN EN ÜSTE ÇIKARILAN GERÇEK KAYNAKLAR:")
print("-" * 70)
for meta in source_metadata_list:
    print(f"[Kaynak {meta['id']}] (Re-rank Skoru: {meta['score']:.4f})")
    print(f"  - Dosya Adı     : {meta['source']}")
    print(f"  - Bölüm (H1)    : {meta['header_1']}")
    print(f"  - Alt Bölüm (H2): {meta['header_2']}")
    print(f"  - Metin Parçası : {meta['snippet']}...")
    print()
print("="*70)

## hibrit

In [ ]:
!pip install rank_bm25

In [ ]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder

# .env dosyasındaki değişkenleri yükle
load_dotenv()

# ==============================================================================
# 1. ADIM: Azure OpenAI, Embedding ve Reranker Modellerini Tanımla
# ==============================================================================
azure_client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)
DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

print("[*] Nomic Embed modeli yükleniyor...")
embeddings = HuggingFaceEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1.5",
    model_kwargs={
        "trust_remote_code": True, 
        "device": "cuda" if os.system("nvidia-smi > /dev/null 2>&1") == 0 else "cpu"
    }
)

print("[*] Reranker modeli (ms-marco-MiniLM-L-6-v2) yükleniyor...")
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# ==============================================================================
# 2. ADIM: Chroma DB'yi Yükle ve Hibrit Arama (Ensemble: BM25 + Vector) Altyapısını Kur
# ==============================================================================
db_dir = "chroma_db_contextual"
print(f"[*] '{db_dir}' dizinindeki Contextual veritabanı yükleniyor...")
db = Chroma(persist_directory=db_dir, embedding_function=embeddings)

# Chroma içindeki tüm dokümanları BM25 (anahtar kelime tabanlı arama) için çekiyoruz
print("[*] Hibrit arama için BM25 (Keyword) indeksi hazırlanıyor...")
all_stored_docs = db.get() # Veritabanındaki tüm metinleri ve metadata'ları al
from langchain_core.documents import Document
bm25_documents = [
    Document(page_content=text, metadata=meta)
    for text, meta in zip(all_stored_docs["documents"], all_stored_docs["metadatas"])
]

# 1. Retriever: Dense Retrieval (Chroma Vektör Arama)
vector_retriever = db.as_retriever(search_kwargs={"k": 20})

# 2. Retriever: Sparse Retrieval (BM25 Kelime Bazlı Arama)
bm25_retriever = BM25Retriever.from_documents(bm25_documents)
bm25_retriever.k = 20

# 3. Ensemble Retriever (Hibrit: %50 Vektör + %50 BM25 Kelime Eşleşmesi)
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

# ==============================================================================
# 3. ADIM: Hibrit Arama ile Adayları Çek
# ==============================================================================
query = "whats the role of americn academy of pediatrics in diabetes care?"
print(f"\n🔎 HİBRİT RAG SORGUSU: '{query}'")

# Hibrit retriever üzerinden adayları çekiyoruz
initial_docs = hybrid_retriever.invoke(query)

# ==============================================================================
# 4. ADIM: RERANKER — Hibrit Adayları Yeniden Sırala
# ==============================================================================
print("[*] Hibrit havuzdan gelen adaylar Reranker ile yeniden puanlanıyor...")

pairs = []
for doc in initial_docs:
    clean_text = doc.page_content.replace("search_document: ", "", 1)
    pairs.append([query, clean_text])

scores = reranker_model.predict(pairs)
scored_docs = list(zip(initial_docs, scores))
scored_docs.sort(key=lambda x: x[1], reverse=True)

# En iyi üst 3 dokümanı seç
top_k_reranked = scored_docs[:3]

# ==============================================================================
# 5. ADIM: Bağlamı Hazırla
# ==============================================================================
context_parts = []
source_metadata_list = []

for idx, (doc, score) in enumerate(top_k_reranked, 1):
    clean_content = doc.page_content.replace("search_document: ", "", 1)
    source = doc.metadata.get('source')
    h1 = doc.metadata.get('header_1')
    h2 = doc.metadata.get('header_2', '')
    
    context_parts.append(f"--- Kaynak [{idx}] (Hibrit + Rerank Skor: {score:.4f}): Dosya: {source} | Başlık: {h1} > {h2} ---\n{clean_content}")
    
    source_metadata_list.append({
        "id": idx,
        "score": float(score),
        "source": source,
        "header_1": h1,
        "header_2": h2,
        "snippet": clean_content
    })

combined_context = "\n\n".join(context_parts)

# ==============================================================================
# 6. ADIM: Azure OpenAI ile Yanıt Üretimi
# ==============================================================================
prompt = f"""
You are a rigorous clinical decision support assistant.

Your task is to answer the user's question using ONLY the information
contained in the provided official medical guideline contexts.

CRITICAL RULES:

1. Do NOT use your pre-trained knowledge or external information.
   Your answer must be supported by the provided contexts.

2. Carefully inspect ALL provided sources before deciding that the
   answer is not available.

3. If the question is directly answered by one or more sources,
   provide that answer clearly and concisely.

4. If the answer is partially supported by the contexts, provide only
   the information that is supported by the contexts. Do not fill the
   missing information using your own knowledge.

5. If the provided contexts genuinely do not contain enough information
   to answer the question, say:
   "The provided guidelines do not contain enough information to answer
   this question."

6. Every factual statement in your answer MUST be followed by the
   source number that supports it, using this format:
   [Source 1], [Source 2], etc.

7. Do NOT cite a source unless the information in that source actually
   supports the statement.

8. Prefer the most direct evidence available in the retrieved context.
   Do not replace a directly stated fact with a general explanation.

[PROVIDED OFFICIAL GUIDELINE CONTEXTS]


{combined_context}



[USER QUESTION]



{query}



[CLINICAL ANSWER]

"""

print("[*] Azure OpenAI ile final klinik yanıt üretiliyor...")
response = azure_client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.0,
    max_tokens=500
)

answer = response.choices[0].message.content.strip()

# ==============================================================================
# 7. ADIM: Sonucu ve Kaynakları Ekrana Bas
# ==============================================================================
print("\n" + "="*70)
print("     AZURE OPENAI + CONTEXTUAL HIBRID RAG + RERANKER YANITI")
print("="*70)
print(answer)
print("="*70)

print("\n🔍 HİBRİT ARAMA VE RERANKER TARAFINDAN SEÇİLEN KANITLAR:")
print("-" * 70)
for meta in source_metadata_list:
    print(f"[Kaynak {meta['id']}] (Skor: {meta['score']:.4f})")
    print(f"  - Dosya Adı     : {meta['source']}")
    print(f"  - Bölüm (H1)    : {meta['header_1']}")
    print(f"  - Alt Bölüm (H2): {meta['header_2']}")
    print(f"  - Metin Parçası : {meta['snippet']}...")
    print()
print("="*70)